# G11.4 - Held-out test report and official-format submission

The test split is scored ONCE, after the validation-based selection. The submission is written in the official template schema (single-band float32 GeoTIFF, [0, 1], template grid and footprint), reopened and validated, with SHA-256 and an experiment manifest. The official DTI evaluator is used only if one is installed (`official_submission.evaluator_command`); otherwise the status is OFFICIAL EVALUATOR UNAVAILABLE. No automatic upload.

Nothing in this notebook trains on test labels, uploads to DrivenData or modifies source data. Every stage is resumable: re-running a cell reuses a valid stage record (`manifests/<stage>.json`).

In [ ]:
# Configuration overrides (everything else: configs/g11_multiscale.yaml). Paths are EXPECTED, verified by the audit.
OVERRIDES = {
    "data_root": "/content/drive/MyDrive/GEMS_2",
    "output_root": "outputs/G11_MULTISCALE_FAULT_DISCOVERY",
    "scratch_root": "/content/g11_scratch",
    "reuse_valid_cache": True,
}

In [ ]:
# Mount Drive, pin the source, install, load the config (no data is copied).
from google.colab import drive
drive.mount("/content/drive")
import subprocess, sys
from pathlib import Path
REPO, REF = "mybenkhadda/GEMS---drivendata", "g11-colab-production"
CANDIDATES = ["/content/GEMS", "/content/GEMS---drivendata", "/content/GEMS_G11"]
CODE = next((c for c in CANDIDATES if (Path(c) / ".git").exists()), CANDIDATES[0])
if not (Path(CODE) / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{REPO}.git", CODE], check=True)
subprocess.run(["git", "-C", CODE, "fetch", "--depth", "1", "origin", REF], check=True)
subprocess.run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], check=True)
if not (Path(CODE) / "gems_bench/g11/pipeline.py").is_file():
    raise RuntimeError(f"{REPO}@{REF} has no gems_bench/g11 - push the G11 package or set REF")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE + "[colab]", "xgboost", "lightgbm"], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
COMMIT = subprocess.check_output(["git", "-C", CODE, "rev-parse", "HEAD"], text=True).strip()
print("checkout", CODE, "commit", COMMIT)

from gems_bench.g11.config import load_config, Mode
from gems_bench.g11.pipeline import G11Pipeline
CFG = load_config(overrides=OVERRIDES)
P = G11Pipeline(CFG)
print("outputs:", P.ctx.root, "| config hash", P.ctx.cfg_hash)
print("Recommended entry point: GEMS_G11_MASTER_COLAB_EXECUTION.ipynb (this notebook runs one phase only)")

In [ ]:
P.audit()
final = P.evaluate(final=True)
display(final[final.split == "test"])

In [ ]:
rec = P.submit()
import json
print(json.dumps({k: rec[k] for k in ("path", "sha256", "validation", "official_evaluation")}, indent=1, default=str))